# Experimento 1: comparación de modelos

En este notebook se entrenan y evalúan cuatro modelos sobre el problema de clasificación de dibujos de gatos y perros: **SVM, Random Forest, una red neuronal MLP y HDBSCAN**.

El objetivo es obtener resultados globales de rendimiento y, además, realizar una pequeña prueba individual con 10 imágenes que fueron reservadas previamente y que no participaron en el entrenamiento, validación ni test.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
import time
from collections import defaultdict

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
import hdbscan
from sklearn.metrics import accuracy_score

## 1. Carga de los datos

Se cargan los conjuntos preparados previamente. El dataset principal contiene los datos de **entrenamiento, validación y test**, mientras que las 10 imágenes reservadas para el experimento individual se cargan desde un archivo independiente.

**Importante:** las imágenes originales de esta prueba deben tener la misma representación utilizada durante el entrenamiento. Como los modelos fueron entrenados con píxeles normalizados en el rango **[0, 1]**, las imágenes individuales se normalizan antes de llegar a esta función.

In [ ]:
# Cargar dataset principal
datos = np.load("datos/gatos_perros_dataset.npz")

X_train = datos["X_train"]
y_train = datos["y_train"]
X_val = datos["X_val"]
y_val = datos["y_val"]
X_test = datos["X_test"]
y_test = datos["y_test"]

# Cargar datos de test por separado
datos_test = np.load("datos/datos-test.npz")

X_test_independiente = datos_test["X_test"]
y_test_independiente = datos_test["y_test"]

# Cargar datos de la prueba individual
datos_individual = np.load("datos/datos-prueba-individual.npz")

X_individual = datos_individual["X_individual"]
y_individual = datos_individual["y_individual"]

X_individual = X_individual.astype(np.float32) / 255.0

print("Datos cargados correctamente:\n")
print(f"X_train:      {X_train.shape} | y_train:      {y_train.shape}")
print(f"X_val:        {X_val.shape} | y_val:        {y_val.shape}")
print(f"X_test:       {X_test.shape} | y_test:       {y_test.shape}")
print(f"X_test_ind.:  {X_test_independiente.shape} | y_test_ind.:  {y_test_independiente.shape}")
print(f"X_individual: {X_individual.shape} | y_individual: {y_individual.shape}")

Datos cargados correctamente:

X_train:      (13993, 784) | y_train:      (13993,)
X_val:        (2998, 784) | y_val:        (2998,)
X_test:       (2999, 784) | y_test:       (2999,)
X_test_ind.:  (2999, 784) | y_test_ind.:  (2999,)
X_individual: (10, 784) | y_individual: (10,)


## 2. Configuración del experimento

Se fijan los parámetros generales del experimento y la semilla aleatoria. Para HDBSCAN se utiliza una muestra de 5.000 imágenes del conjunto de entrenamiento, manteniendo su correspondiente etiqueta real para poder evaluar posteriormente los clusters.

In [3]:
RANDOM_STATE = 42
HDBSCAN_TRAIN_SIZE = 5_000

In [4]:
rng = np.random.default_rng(RANDOM_STATE)
hdb_idx = rng.choice(len(X_train), HDBSCAN_TRAIN_SIZE, replace=False)

X_hdbscan = X_train[hdb_idx]
y_hdbscan = y_train[hdb_idx]

## 3. Funciones auxiliares

A continuación se definen funciones para mostrar los resultados de los modelos y para realizar la prueba individual sobre las imágenes reservadas.

In [5]:
def print_resultado(resultados:dict) -> None:
    print(f"\n{'=' * 60}")
    print(resultados["modelo"])
    print("=" * 60)

    print(f"Tiempo entrenamiento: {resultados["tiempo_entrenamiento_s"]:.3f} s")
    print(f"Tiempo predicción:    {resultados["tiempo_prediccion_s"]:.3f} s")
    print(f"Accuracy:             {resultados['accuracy']:.4f}")

### Prueba individual

La prueba individual mide cuánto tarda cada modelo en clasificar cada una de las 10 imágenes reservadas y qué etiqueta asigna.

In [ ]:
def evaluar_imagenes_singulares(modelos, X_individual, nombres_modelos):
    resultados = []

    for modelo, nombre in zip(modelos, nombres_modelos):

        for i, imagen in enumerate(X_individual):

            imagen_input = imagen.reshape(1, -1)

            inicio = time.perf_counter()

            if nombre == "HDBSCAN":

                cluster, _ = hdbscan.approximate_predict(
                    modelo,
                    imagen_input
                )

                cluster = cluster[0]

                if cluster == -1:
                    etiqueta = -1

                elif cluster in modelo.cluster_a_clase_:
                    etiqueta = modelo.cluster_a_clase_[cluster]

                else:
                    etiqueta = -1

            else:
                etiqueta = modelo.predict(imagen_input)[0]

            tiempo_prediccion = time.perf_counter() - inicio

            imagen_uint8 = (imagen * 255).astype(np.uint8)
            imagen = imagen_uint8.reshape(28, 28)

            resultados.append({
                "modelo": nombre,
                "indice_imagen": i,
                "imagen": imagen.copy(),
                "etiqueta": etiqueta,
                "tiempo_prediccion_s": tiempo_prediccion
            })

    return resultados

### Evaluación de clasificadores

Esta función entrena cada clasificador, mide su tiempo de entrenamiento, mide el tiempo de predicción sobre el conjunto de test y calcula su **Accuracy**. Finalmente, utiliza las mismas imágenes reservadas para realizar el experimento individual.

In [ ]:
def evaluar_clasificador(nombre, modelo, X_individual=None):

    # Entrenamiento

    inicio = time.perf_counter()

    modelo.fit(X_train, y_train)

    tiempo_entrenamiento = time.perf_counter() - inicio

    # Prediccion

    inicio = time.perf_counter()

    pred = modelo.predict(X_test)

    tiempo_prediccion = time.perf_counter() - inicio

    # Resultados

    resultados = {
        "modelo": nombre,
        "tiempo_entrenamiento_s": tiempo_entrenamiento,
        "tiempo_prediccion_s": tiempo_prediccion,
        "accuracy": accuracy_score(y_test, pred)
    }

    print_resultado(resultados)

    # Experimento

    resultados_individuales = None

    if X_individual is not None:
        resultados_individuales = evaluar_imagenes_singulares(
            [modelo],
            X_individual,
            [nombre]
        )

    return modelo, pred, resultados, resultados_individuales

### Evaluación de HDBSCAN

HDBSCAN se utiliza como algoritmo de clustering. Como sus clusters no tienen directamente las etiquetas `gato` y `perro`, se construye una correspondencia entre cada cluster y la clase mayoritaria observada durante el entrenamiento. De esta forma podemos obtener una Accuracy comparable para este experimento.

In [ ]:
def entrenar_hdbscan(X_train, y_train, X_test, y_test,
                     X_individual=None):

    modelo = hdbscan.HDBSCAN(
        min_cluster_size=50,
        min_samples=10,
        metric="euclidean",
        cluster_selection_method="eom",
        prediction_data=True
    )

    # -------------------------
    # Entrenamiento
    # -------------------------

    inicio = time.perf_counter()

    modelo.fit(X_train)

    tiempo_entrenamiento = time.perf_counter() - inicio

    # -------------------------
    # Crear mapa: cluster -> clase
    # -------------------------

    clusters = modelo.labels_

    mapa_clusters = defaultdict(list)

    for cluster, clase in zip(clusters, y_train):

        if cluster != -1:
            mapa_clusters[cluster].append(clase)

    cluster_a_clase = {}

    for cluster, clases in mapa_clusters.items():

        valores, cantidades = np.unique(
            clases,
            return_counts=True
        )

        clase_mayoritaria = valores[np.argmax(cantidades)]

        cluster_a_clase[cluster] = clase_mayoritaria

    # Guardamos el mapa dentro del modelo para utilizarlo
    # posteriormente en las predicciones individuales.

    modelo.cluster_a_clase_ = cluster_a_clase

    # -------------------------
    # Prediccion sobre TEST
    # -------------------------

    inicio = time.perf_counter()

    clusters_predichos, _ = hdbscan.approximate_predict(
        modelo,
        X_test
    )

    tiempo_prediccion = time.perf_counter() - inicio

    y_pred = []

    for cluster in clusters_predichos:

        if cluster == -1:
            y_pred.append(-1)

        elif cluster in cluster_a_clase:
            y_pred.append(cluster_a_clase[cluster])

        else:
            y_pred.append(-1)

    y_pred = np.array(y_pred)

    # -------------------------
    # Resultados globales
    # -------------------------

    resultados = {
        "modelo": "HDBSCAN",
        "accuracy": accuracy_score(y_test, y_pred),
        "tiempo_entrenamiento_s": tiempo_entrenamiento,
        "tiempo_prediccion_s": tiempo_prediccion
    }

    print_resultado(resultados)

    # -------------------------
    # Experimento individual
    # -------------------------

    resultados_individuales = None

    if X_individual is not None:
        resultados_individuales = evaluar_imagenes_singulares(
            [modelo],
            X_individual,
            ["HDBSCAN"]
        )

    return modelo, resultados, resultados_individuales

## 4. Entrenamiento y evaluación de los modelos

Se inicializan los cuatro modelos con los parámetros definidos para este experimento y se ejecutan las funciones de evaluación.

### Preparación de la prueba individual

Las imágenes reservadas fueron guardadas originalmente como píxeles entre 0 y 255. Antes de entregarlas a los modelos, se convierten a `float32` y se dividen por 255 para obtener exactamente el mismo rango `[0, 1]` utilizado durante el entrenamiento.

In [ ]:
svm_model = SVC(
    kernel="rbf",
    C=10,
    gamma="scale",
    cache_size=2048
)

svm_model, svm_pred, svm_results, svm_individuales = evaluar_clasificador(
    "SVM",
    svm_model,
    X_individual
)

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    max_features="sqrt",
    n_jobs=-1,
    random_state=42
)

rf_model, rf_pred, rf_results, rf_individuales = evaluar_clasificador(
    "Random Forest",
    rf_model,
    X_individual
)

In [ ]:
mlp_model = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    solver="adam",
    batch_size=128,
    learning_rate_init=0.001,
    max_iter=30,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=5,
    random_state=42
)

mlp_model, mlp_pred, mlp_results, mlp_individuales = evaluar_clasificador(
    "Red neuronal MLP",
    mlp_model,
    X_individual
)

In [ ]:
hdbscan_model, hdbscan_results, hdbscan_individuales = entrenar_hdbscan(
    X_train,
    y_train,
    X_test,
    y_test,
    X_individual
)

## 5. Resultados globales

Se reúnen los resultados de los cuatro modelos en un único DataFrame y se guardan en CSV. Estos son los datos que posteriormente pueden utilizarse para comparar Accuracy y tiempos de ejecución en la visualización.

In [ ]:
variables_resultados = [
    svm_results,
    rf_results,
    mlp_results,
    hdbscan_results
]

df_resultados_globales = pd.DataFrame(variables_resultados)

df_resultados_globales.to_csv(
    "resultados/resultados_globales.csv",
    index=False
)

## 6. Resultados de la prueba individual

Finalmente se reúnen los resultados de las 10 imágenes para los cuatro modelos. Se conservan por separado las imágenes y la información de cada predicción, incluyendo el modelo, la etiqueta asignada y el tiempo de predicción.

In [ ]:
resultados_individuales = (
    svm_individuales
    + rf_individuales
    + mlp_individuales
    + hdbscan_individuales
)

imagenes_individuales = np.array([
    resultado["imagen"]
    for resultado in resultados_individuales
])

df_resultados_individuales = pd.DataFrame([
    {
        "modelo": resultado["modelo"],
        "indice_imagen": resultado["indice_imagen"],
        "etiqueta": resultado["etiqueta"],
        "tiempo_prediccion_s": resultado["tiempo_prediccion_s"]
    }
    for resultado in resultados_individuales
])

np.savez_compressed(
    "resultados/experimento_individual_imagenes.npz",
    imagenes=imagenes_individuales
)

df_resultados_individuales.to_csv(
    "resultados/experimento_individual_resultados.csv",
    index=False
)